In [13]:
# 加载环境变量
from dotenv import load_dotenv
from langchain_core.messages import AIMessage

load_dotenv()

True

# 定义模型
我们定义两个模型，一个多模态模型，一个聊天模型


In [14]:
from langchain.chat_models import init_chat_model
import os

# 多模态模型
multimodal_model = init_chat_model(
    model="qwen3.5-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 定义工具


In [15]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_results=5,
    topic="general"
)

# 添加记忆管理

这里我们采用sqllite

In [16]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 初始化checkpointer
checkpointer = SqliteSaver(sqlite3.connect("resources/personal_chief.db", check_same_thread=False))
# 自动建表
checkpointer.setup()

# 定义智能体

In [17]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，再搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model=multimodal_model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

# 测试


In [22]:
from langchain.messages import HumanMessage

multimodal_message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {"url": "https://img95.699pic.com/photo/60053/9699.jpg_wh860.jpg"},
        },
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "6"}}

In [23]:
response = agent.invoke({"messages": [multimodal_message]}, config)

In [24]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image_url', 'image_url': {'url': 'https://img95.699pic.com/photo/60053/9699.jpg_wh860.jpg'}}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_90f05b46327b4fd0b9e9ab2c)
 Call ID: call_90f05b46327b4fd0b9e9ab2c
  Args:
    query: 混合蔬菜食谱 西兰花 胡萝卜 彩椒 土豆 健康菜谱
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "混合蔬菜食谱 西兰花 胡萝卜 彩椒 土豆 健康菜谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.reddit.com/r/cookingforbeginners/comments/jal8ba/eli5_how_do_i_cook_broccoli_carrots_and_potatoes?tl=zh-hans", "title": "简单来说：怎么做西兰花、胡萝卜和土豆？", "content": "烤大蒜黄油土豆、西兰花和胡萝卜。将1/3杯融化的黄油、1汤匙切碎的大蒜、1茶匙欧芹、1/2茶匙百里香、1茶匙盐和1/4茶匙黑胡椒混合在碗中，然后加入蔬菜并", "score": 0.89121616, "raw_content": null, "id": 

In [25]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第3道菜，可以说的更详细点吗？")]},
    config
)

In [26]:
# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

# 🥣 奶香花椰菜浓汤 - 详细烹饪指南

## 📋 食材清单（4-6人份）

### 核心食材（您照片中都有✓）
| 食材 | 用量 | 处理建议 |
|------|------|----------|
| 花椰菜/西兰花 | 2大颗 | 取花朵部分，茎部去皮切片也可用 |
| 土豆 | 1大个 | 去皮切小块，增加浓稠度 |
| 胡萝卜 | 1根 | 去皮切片，增加甜味 |
| 洋葱 | 1个 | 切丁，炒香后更美味 |
| 芹菜 | 2根 | 切丁，增加清香 |
| 大蒜 | 3-4瓣 | 切片或剁碎 |

### 调味料（需自备）
| 食材 | 用量 | 替代方案 |
|------|------|----------|
| 食用油 | 1½茶匙 | 橄榄油/菜籽油均可 |
| 牛奶 | 1杯 | 可用豆浆/椰奶替代 |
| 鸡汤/蔬菜汤 | 2罐(约700ml) | 清水+鸡精也可 |
| 盐 | ¾茶匙 | 根据口味调整 |
| 黑胡椒 | ¼茶匙 | 现磨更香 |
| 奶酪（可选） | 30g | 切达奶酪最佳 |

---

## 👨‍ 详细步骤（总耗时约55分钟）

### 第一阶段：备菜（15分钟）
```
步骤1️⃣ 清洗所有蔬菜
步骤2️⃣ 洋葱、芹菜切丁（约0.5cm大小）
步骤3️⃣ 土豆、胡萝卜去皮切薄片
步骤4️⃣ 花椰菜切成小朵，茎部单独切片
步骤5️⃣ 大蒜剁碎备用
```

### 第二阶段：炒香底料（10分钟）
```
步骤6️⃣ 大锅中火加热油
步骤7️ 加入洋葱+芹菜，炒6-8分钟至透明金黄
步骤8️⃣ 加入大蒜，再炒30秒（⚠️不要炒焦）
```

### 第三阶段：炖煮（25分钟）
```
步骤9️⃣ 加入土豆、胡萝卜、花椰菜茎、月桂叶（如有）
步骤🔟 倒入牛奶和鸡汤，大火煮沸
步骤1️⃣1️⃣ 转小火煮15分钟至蔬菜变软
步骤1️⃣2️⃣ 最后10分钟加入花椰菜花朵（保持翠绿色）
```

### 第四阶段：调味完成（5分钟）
```
步骤1️⃣3️⃣ 取出月桂叶丢弃
步骤1️4️⃣ 用搅拌机打碎一半汤（保留部分蔬菜口感）
步骤1️⃣5️⃣ 加入奶酪、盐

# 流式调用

有工具调用的agent消息类型比较多，要正确的流式输出比较麻烦，这里给出一个示例


In [27]:
from langchain.messages import AIMessageChunk

for chunk, metadata in agent.stream(
            {"messages": [multimodal_message]},
            config,
            stream_mode="messages"
        ):
            if isinstance(chunk, AIMessageChunk) and chunk.content:
                print(chunk.content, end="", flush=True)

# 🥣 奶香花椰菜浓汤 - 详细烹饪指南

很高兴您选择了这道汤！它确实是秋冬季节的暖心首选。以下是完整的制作详解：

---

## 📋 食材清单（4-6人份）

### 核心食材（您照片中都有✓）
| 食材 | 用量 | 处理建议 |
|------|------|----------|
| 花椰菜/西兰花 | 2大颗 | 取花朵部分，茎部去皮切片也可用 |
| 土豆 | 1大个 | 去皮切小块，增加浓稠度 |
| 胡萝卜 | 1根 | 去皮切片，增加甜味 |
| 洋葱 | 1个 | 切丁，炒香后更美味 |
| 芹菜 | 2根 | 切丁，增加清香 |
| 大蒜 | 3-4瓣 | 切片或剁碎 |

### 调味料（需自备）
| 食材 | 用量 | 替代方案 |
|------|------|----------|
| 食用油 | 1½茶匙 | 橄榄油/菜籽油均可 |
| 牛奶 | 1杯 | 可用豆浆/椰奶替代 |
| 鸡汤/蔬菜汤 | 2罐(约700ml) | 清水+鸡精也可 |
| 盐 | ¾茶匙 | 根据口味调整 |
| 黑胡椒 | ¼茶匙 | 现磨更香 |
| 奶酪（可选） | 30g | 切达奶酪最佳 |

---

## ‍ 详细步骤（总耗时约55分钟）

### 第一阶段：备菜（15分钟）
```
步骤1️⃣ 清洗所有蔬菜
步骤2️⃣ 洋葱、芹菜切丁（约0.5cm大小）
步骤3️⃣ 土豆、胡萝卜去皮切薄片
步骤4️⃣ 花椰菜切成小朵，茎部单独切片
步骤5️⃣ 大蒜剁碎备用
```

### 第二阶段：炒香底料（10分钟）
```
步骤6️⃣ 大锅中火加热油
步骤7️⃣ 加入洋葱+芹菜，炒6-8分钟至透明金黄
步骤8️⃣ 加入大蒜，再炒30秒（⚠️不要炒焦）
```

### 第三阶段：炖煮（25分钟）
```
步骤9️⃣ 加入土豆、胡萝卜、花椰菜茎、月桂叶（如有）
步骤🔟 倒入牛奶和鸡汤，大火煮沸
步骤1️⃣1️⃣ 转小火煮15分钟至蔬菜变软
步骤1️⃣2️⃣ 最后10分钟加入花椰菜花朵（保持翠绿色）
```

### 第四阶段：调味完成（5分钟）
```
步骤1️⃣3️⃣ 取出月桂叶丢弃
步骤1️4️⃣ 用搅拌机打碎一半汤（保留部分蔬菜口感）
步骤1️⃣5️⃣ 加入奶酪、盐、胡椒，小火搅拌2-3分钟
步骤1️⃣6️⃣ 装盘，可撒少许香草装饰
```


# 获取会话历史

既然会话记忆交给了checkpointer管理，我们自然可以从中获取会话历史


In [28]:
checkpointer.get(config)

{'v': 4,
 'ts': '2026-09-16T07:10:47.004753+00:00',
 'id': '1f1b19db-d5b2-6f9b-8009-1dc2b036257d',
 'channel_values': {'messages': [HumanMessage(content=[{'type': 'image_url', 'image_url': {'url': 'https://img95.699pic.com/photo/60053/9699.jpg_wh860.jpg'}}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}], additional_kwargs={}, response_metadata={}, id='935b730d-a9f0-4b7b-abe4-cb84c4d49f2c'),
   AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 694, 'prompt_tokens': 2481, 'total_tokens': 3175, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 637, 'rejected_prediction_tokens': None, 'text_tokens': 57}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': None, 'image_tokens': 488, 'text_tokens': 1993}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus', 'system_fingerprint': None, 'id': 'chatcmpl-0fd0bb7e-230d-9b38-96d2-5ed

In [29]:
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

<class 'langchain_core.messages.human.HumanMessage'>
content=[{'type': 'image_url', 'image_url': {'url': 'https://img95.699pic.com/photo/60053/9699.jpg_wh860.jpg'}}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}] additional_kwargs={} response_metadata={} id='935b730d-a9f0-4b7b-abe4-cb84c4d49f2c'
<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 694, 'prompt_tokens': 2481, 'total_tokens': 3175, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 637, 'rejected_prediction_tokens': None, 'text_tokens': 57}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': None, 'image_tokens': 488, 'text_tokens': 1993}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-plus', 'system_fingerprint': None, 'id': 'chatcmpl-0fd0bb7e-230d-9b38-96d2-5ed640f245ce', 'finish_reason': 'tool_calls', 'logprobs': None} id='

定义一个便捷的获取会话历史的函数：


In [30]:
def get_messages(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""

    # 根据 thread_id 查询 checkpoint
    cp = checkpointer.get({"configurable": {"thread_id": thread_id}})

    # 如果不存在，返回空列表
    if not cp:
        return []

    # 安全获取 messages
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []

    messages = channel_values.get("messages", [])
    if not messages:
        return []

    # 转换消息格式
    result = []
    for msg in messages:
        if not msg.content:
            continue

        if isinstance(msg, HumanMessage):
            result.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"role": "assistant", "content": msg.content})

    return result

In [31]:
get_messages("1")

[{'role': 'user',
  'content': [{'type': 'image',
    'url': 'https://tmp9527.oss-cn-beijing.aliyuncs.com/1772678117299.jpeg?x-oss-signature-version=OSS4-HMAC-SHA256&x-oss-date=20260305T082142Z&x-oss-expires=299&x-oss-credential=LTAI5tHzyC36KhUJfPqibaCo%2F20260305%2Fcn-beijing%2Foss%2Faliyun_v4_request&x-oss-signature=90083bdacfacc36386069d1e82c33b04ccd8eaa71c43fcf9248a36535d9cf46a'},
   {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'url': 'https://tmp9527.oss-cn-beijing.aliyuncs.com/1772678117299.jpeg'},
   {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'url': 'https://tmp9527.oss-cn-beijing.aliyuncs.com/1772678117299.jpeg?x-oss-signature-version=OSS4-HMAC-SHA256&x-oss-date=20260305T082142Z&x-oss-expires=299&x-oss-credential=LTAI5tHzyC36KhUJfPqibaCo%2F20260305%2Fcn-beijing%2Foss%2Faliyun_v4_request&x-oss-signature=90083bdacfacc36386069d1e82c33b04ccd8eaa71c43fcf9248a36535d9cf46a'}

# 清空会话历史


In [39]:
checkpointer.delete_thread("1")